# Surface Wind Speed (sfcwind)

#### data import

In [1]:
import intake
import numpy as np
import matplotlib.pylab as plt
import xarray as xr
import matplotlib.colors as colors
from shapely.geometry import Polygon, Point, box
from shapely import contains_xy
import time
from scipy.stats import linregress
import sys
sys.path.append("/home/b/b383696/02_Functions")
from function_area_Rradii_shift_new_Version import eddyR_donut_sst
from function_area_Rradii_shift_new_Version import eddyR_donut_shift_var
from function_donut import dif_mean_calculation

In [2]:
#import data with ice free 3R (and no land mass)
edso_ac = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso_ac_3R_0_ocean.nc")
edso_c = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso_c_3R_0_ocean.nc")

In [3]:
eerie_cat=intake.open_catalog("https://raw.githubusercontent.com/eerie-project/intake_catalogues/main/eerie.yaml")
da_atmos = eerie_cat["dkrz"]["disk"]["model-output"]["icon-esm-er"]["hist-1950"]["v20240618"]["atmos"]["gr025"]["2d_daily_mean"].to_dask()
da_ocean = eerie_cat["dkrz"]["disk"]["model-output"]["icon-esm-er"]["hist-1950"]["v20240618"]["ocean"]["gr025"]["2d_daily_mean"].to_dask()

In [4]:
#select atmos and ocean data for SO
da_atmos_so = da_atmos.where(da_atmos.lat < -50, drop = True)
da_ocean_so = da_ocean.where(da_ocean.lat < -50, drop = True)

In [5]:
#import data
sst1_ac = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/sst1_dn23_ac.nc")
sst1_c = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/sst1_dn23_c.nc")

In [5]:
#reduce sfcwind by the dimension height_2
da_atmos_so_sfcwind = da_atmos_so.sfcwind.squeeze(dim="height_2")

In [10]:
da_atmos_so_sfcwind

NameError: name 'da_atmos_so_sfcwind' is not defined

In [19]:
#export data
sfcwind1_dn23_ac = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/sfcwind1_dn23_ac.nc")
sfcwind1_dn23_c = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/sfcwind1_dn23_c.nc")

In [6]:
#import data
edso1_dn23_sfcwind_ac = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_ac.nc")
edso1_dn23_sfcwind_c = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_c.nc")

In [18]:
sfcwind1_dn23_ac

<xarray.Dataset>
Dimensions:                (obs: 1741153)
Coordinates:
  * obs                    (obs) int64 0 1 2 3 ... 1741150 1741151 1741152
Data variables:
    sfcwind_mean_ed        (obs) float64 ...
    sfcwind_npoints_ed     (obs) int64 ...
    sfcwind_mean_donut     (obs) float64 ...
    sfcwind_npoints_donut  (obs) int64 ...
    ID                     (obs) float64 ...

In [7]:
edso1_dn23_sfcwind_q75_e20 = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_q75_e20.nc")

In [11]:
#import data
edso1_sfcwind_ac_q75 = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_ac_q75.nc")
edso1_sfcwind_c_q75 = xr.open_dataset("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_c_q75.nc")

#### calculate dif_sst with 1R and donut 2-3

In [5]:
sst1_ac = eddyR_donut_sst(da_ocean_so, edso_ac, 3, 2, 1)

In [6]:
sst1_c = eddyR_donut_sst(da_ocean_so, edso_c, 3, 2, 1)

In [8]:
#### select warm AC and cold C (no shift, all data)
sst1_ac_warm = sst1_ac.where(sst1_ac.dif_sst > 0, drop = True)
sst1_ac_cold = sst1_ac.where(sst1_ac.dif_sst < 0, drop = True)
sst1_c_warm = sst1_c.where(sst1_c.dif_sst > 0, drop = True)
sst1_c_cold = sst1_c.where(sst1_c.dif_sst < 0, drop = True)

In [9]:
sst1_ac_warm

<xarray.Dataset>
Dimensions:            (obs: 1179827)
Coordinates:
  * obs                (obs) int64 3 4 5 6 7 ... 1741145 1741146 1741147 1741148
Data variables:
    sst_mean_ed        (obs) float64 4.328 4.242 3.898 ... 6.27 6.486 6.386
    sst_npoints_ed     (obs) float64 6.0 6.0 6.0 5.0 6.0 ... 4.0 2.0 3.0 3.0 3.0
    sst_mean_donut     (obs) float64 3.945 3.971 3.807 ... 5.986 6.036 6.109
    sst_npoints_donut  (obs) float64 29.0 39.0 34.0 20.0 ... 12.0 13.0 13.0 13.0
    dif_sst            (obs) float64 0.3831 0.2716 0.09163 ... 0.4498 0.2765
    ID                 (obs) float64 4.0 5.0 6.0 ... 1.987e+06 1.987e+06

In [10]:
sst1_ac_cold

<xarray.Dataset>
Dimensions:            (obs: 561314)
Coordinates:
  * obs                (obs) int64 0 1 2 11 ... 1741149 1741150 1741151 1741152
Data variables:
    sst_mean_ed        (obs) float64 2.889 3.52 3.978 ... 1.756 1.839 2.202
    sst_npoints_ed     (obs) float64 4.0 4.0 4.0 27.0 24.0 ... 3.0 3.0 4.0 2.0
    sst_mean_donut     (obs) float64 3.223 3.798 4.057 ... 2.519 2.469 2.873
    sst_npoints_donut  (obs) float64 16.0 17.0 20.0 127.0 ... 19.0 17.0 14.0
    dif_sst            (obs) float64 -0.3339 -0.2778 -0.07883 ... -0.63 -0.6715
    ID                 (obs) float64 1.0 2.0 3.0 ... 1.987e+06 1.987e+06

In [11]:
sst1_c_warm

<xarray.Dataset>
Dimensions:            (obs: 559278)
Coordinates:
  * obs                (obs) int64 3 4 5 6 7 ... 1661588 1661589 1661590 1661591
Data variables:
    sst_mean_ed        (obs) float64 2.297 2.211 2.102 ... 2.984 2.843 2.898
    sst_npoints_ed     (obs) float64 6.0 3.0 4.0 3.0 3.0 ... 4.0 5.0 8.0 7.0
    sst_mean_donut     (obs) float64 2.179 2.167 2.088 ... 2.959 2.817 2.86
    sst_npoints_donut  (obs) float64 26.0 14.0 18.0 13.0 ... 24.0 26.0 38.0 31.0
    dif_sst            (obs) float64 0.1187 0.04379 0.01432 ... 0.02598 0.03798
    ID                 (obs) float64 12.0 15.0 16.0 ... 1.95e+06 1.95e+06

In [12]:
sst1_c_cold

<xarray.Dataset>
Dimensions:            (obs: 1102293)
Coordinates:
  * obs                (obs) int64 0 1 2 17 ... 1661600 1661601 1661602 1661603
Data variables:
    sst_mean_ed        (obs) float64 2.079 2.005 2.239 ... 5.254 5.442 5.523
    sst_npoints_ed     (obs) float64 8.0 8.0 4.0 5.0 6.0 ... 50.0 51.0 48.0 48.0
    sst_mean_donut     (obs) float64 2.201 2.053 2.254 ... 5.419 5.643 5.634
    sst_npoints_donut  (obs) float64 40.0 34.0 22.0 20.0 ... 256.0 238.0 254.0
    dif_sst            (obs) float64 -0.1219 -0.04753 ... -0.2014 -0.1106
    ID                 (obs) float64 8.0 9.0 11.0 ... 1.95e+06 1.95e+06 1.95e+06

In [13]:
#export data
sst1_ac.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/sst1_dn23_ac.nc")
sst1_c.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/sst1_dn23_c.nc")

#### calculate dif_sfcwind with 1R and donut 2-3

In [8]:
sfcwind1_dn23_ac = eddyR_donut_shift_var(da_atmos_so_sfcwind, edso_ac, "sfcwind", 3, 2, 0, 1)

In [9]:
sfcwind1_dn23_c = eddyR_donut_shift_var(da_atmos_so_sfcwind, edso_c, "sfcwind", 3, 2, 0, 1)

In [10]:
#export data
sfcwind1_dn23_ac.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/sfcwind1_dn23_ac.nc")
sfcwind1_dn23_c.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/sfcwind1_dn23_c.nc")

#### combine and calculate dif_sfcwind

In [11]:
edso1_dn23_sfcwind_ac = dif_mean_calculation(edso_ac, sst1_ac, sfcwind1_dn23_ac, "sfcwind", 0)

In [12]:
edso1_dn23_sfcwind_c = dif_mean_calculation(edso_c, sst1_c, sfcwind1_dn23_c, "sfcwind", 0)

In [13]:
edso1_dn23_sfcwind_ac

<xarray.Dataset>
Dimensions:                        (obs: 1741141, NbSample: 50)
Coordinates:
  * obs                            (obs) float64 1.0 2.0 ... 1.987e+06 1.987e+06
Dimensions without coordinates: NbSample
Data variables: (12/41)
    amplitude                      (obs) float64 0.008 0.01 ... 0.0324 0.0275
    cost_association               (obs) float32 0.2152 0.2833 ... 9.969e+36
    effective_area                 (obs) float32 5.428e+08 ... 5.388e+08
    effective_contour_height       (obs) float32 -0.048 -0.048 ... 0.188 0.196
    effective_contour_latitude     (obs, NbSample) float64 -75.94 ... -69.06
    effective_contour_longitude    (obs, NbSample) float64 203.5 203.6 ... 12.75
    ...                             ...
    dif_sfcwind                    (obs) float64 -0.6381 0.0295 ... 0.01385
    sst_mean_ed                    (obs) float64 2.889 3.52 ... 1.839 2.202
    sst_npoints_ed                 (obs) float64 4.0 4.0 4.0 6.0 ... 3.0 4.0 2.0
    sst_mean_donut                 (obs) float64 3.223 3.798 ... 2.469 2.873
    sst_npoints_donut              (obs) float64 16.0 17.0 20.0 ... 17.0 14.0
    dif_sst                        (obs) float64 -0.3339 -0.2778 ... -0.6715
Attributes: (12/13)
    track_extra_variables:     lat_max,lon_max
    track_array_variables:     50
    array_variables:           contour_lat_e,contour_lon_e,contour_lat_s,cont...
    title:                     Anticyclonic
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    ...                        ...
    framework_version:         0+unknown
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    date_created:              2025-09-05T07:58:08Z
    time_coverage_duration:    P8034D
    time_coverage_start:       1993-01-01T00:00:00Z
    time_coverage_end:         2014-12-31T00:00:00Z

In [14]:
#export data
edso1_dn23_sfcwind_ac.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_ac.nc")
edso1_dn23_sfcwind_c.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_c.nc")

#### select warm/cold eddies

In [15]:
#### select warm AC and cold C (no shift, all data)
edso1_dn23_sfcwind_ac_warm = edso1_dn23_sfcwind_ac.where(edso1_dn23_sfcwind_ac.dif_sst > 0, drop = True)
edso1_dn23_sfcwind_ac_cold = edso1_dn23_sfcwind_ac.where(edso1_dn23_sfcwind_ac.dif_sst < 0, drop = True)
edso1_dn23_sfcwind_c_warm = edso1_dn23_sfcwind_c.where(edso1_dn23_sfcwind_c.dif_sst > 0, drop = True)
edso1_dn23_sfcwind_c_cold = edso1_dn23_sfcwind_c.where(edso1_dn23_sfcwind_c.dif_sst < 0, drop = True)

In [16]:
edso1_dn23_sfcwind_ac_warm

<xarray.Dataset>
Dimensions:                        (obs: 1179827, NbSample: 50)
Coordinates:
  * obs                            (obs) float64 4.0 5.0 ... 1.987e+06 1.987e+06
Dimensions without coordinates: NbSample
Data variables: (12/41)
    amplitude                      (obs) float64 0.0173 0.0198 ... 0.0091 0.0092
    cost_association               (obs) float32 0.2485 0.1088 ... 9.969e+36
    effective_area                 (obs) float32 1.226e+09 ... 7.081e+08
    effective_contour_height       (obs) float32 -0.048 -0.048 ... 0.078 0.08
    effective_contour_latitude     (obs, NbSample) float64 -75.99 ... -69.04
    effective_contour_longitude    (obs, NbSample) float64 203.0 203.2 ... 285.8
    ...                             ...
    dif_sfcwind                    (obs) float64 0.1122 0.1288 ... 0.0403
    sst_mean_ed                    (obs) float64 4.328 4.242 ... 6.486 6.386
    sst_npoints_ed                 (obs) float64 6.0 6.0 6.0 5.0 ... 3.0 3.0 3.0
    sst_mean_donut                 (obs) float64 3.945 3.971 ... 6.036 6.109
    sst_npoints_donut              (obs) float64 29.0 39.0 34.0 ... 13.0 13.0
    dif_sst                        (obs) float64 0.3831 0.2716 ... 0.4498 0.2765
Attributes: (12/13)
    track_extra_variables:     lat_max,lon_max
    track_array_variables:     50
    array_variables:           contour_lat_e,contour_lon_e,contour_lat_s,cont...
    title:                     Anticyclonic
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    ...                        ...
    framework_version:         0+unknown
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    date_created:              2025-09-05T07:58:08Z
    time_coverage_duration:    P8034D
    time_coverage_start:       1993-01-01T00:00:00Z
    time_coverage_end:         2014-12-31T00:00:00Z

In [17]:
edso1_dn23_sfcwind_ac_cold

<xarray.Dataset>
Dimensions:                        (obs: 561314, NbSample: 50)
Coordinates:
  * obs                            (obs) float64 1.0 2.0 ... 1.987e+06 1.987e+06
Dimensions without coordinates: NbSample
Data variables: (12/41)
    amplitude                      (obs) float64 0.008 0.01 ... 0.0324 0.0275
    cost_association               (obs) float32 0.2152 0.2833 ... 9.969e+36
    effective_area                 (obs) float32 5.428e+08 ... 5.388e+08
    effective_contour_height       (obs) float32 -0.048 -0.048 ... 0.188 0.196
    effective_contour_latitude     (obs, NbSample) float64 -75.94 ... -69.06
    effective_contour_longitude    (obs, NbSample) float64 203.5 203.6 ... 12.75
    ...                             ...
    dif_sfcwind                    (obs) float64 -0.6381 0.0295 ... 0.01385
    sst_mean_ed                    (obs) float64 2.889 3.52 ... 1.839 2.202
    sst_npoints_ed                 (obs) float64 4.0 4.0 4.0 ... 3.0 4.0 2.0
    sst_mean_donut                 (obs) float64 3.223 3.798 ... 2.469 2.873
    sst_npoints_donut              (obs) float64 16.0 17.0 20.0 ... 17.0 14.0
    dif_sst                        (obs) float64 -0.3339 -0.2778 ... -0.6715
Attributes: (12/13)
    track_extra_variables:     lat_max,lon_max
    track_array_variables:     50
    array_variables:           contour_lat_e,contour_lon_e,contour_lat_s,cont...
    title:                     Anticyclonic
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    ...                        ...
    framework_version:         0+unknown
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    date_created:              2025-09-05T07:58:08Z
    time_coverage_duration:    P8034D
    time_coverage_start:       1993-01-01T00:00:00Z
    time_coverage_end:         2014-12-31T00:00:00Z

In [18]:
edso1_dn23_sfcwind_c_warm

<xarray.Dataset>
Dimensions:                        (obs: 559278, NbSample: 50)
Coordinates:
  * obs                            (obs) float64 12.0 15.0 ... 1.95e+06 1.95e+06
Dimensions without coordinates: NbSample
Data variables: (12/41)
    amplitude                      (obs) float64 -0.0221 -0.0408 ... -0.016
    cost_association               (obs) float32 0.2426 0.1914 ... 9.969e+36
    effective_area                 (obs) float32 8.83e+08 8.44e+08 ... 1.943e+09
    effective_contour_height       (obs) float32 0.04 0.036 ... 6.661e-16 0.012
    effective_contour_latitude     (obs, NbSample) float64 -70.02 ... -64.08
    effective_contour_longitude    (obs, NbSample) float64 356.5 356.5 ... 183.0
    ...                             ...
    dif_sfcwind                    (obs) float64 0.4903 0.5562 ... -0.1256
    sst_mean_ed                    (obs) float64 2.297 2.211 ... 2.843 2.898
    sst_npoints_ed                 (obs) float64 6.0 3.0 4.0 3.0 ... 5.0 8.0 7.0
    sst_mean_donut                 (obs) float64 2.179 2.167 ... 2.817 2.86
    sst_npoints_donut              (obs) float64 26.0 14.0 18.0 ... 38.0 31.0
    dif_sst                        (obs) float64 0.1187 0.04379 ... 0.03798
Attributes: (12/13)
    track_extra_variables:     lat_max,lon_max
    track_array_variables:     50
    array_variables:           contour_lat_e,contour_lon_e,contour_lat_s,cont...
    title:                     Cyclonic
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    ...                        ...
    framework_version:         0+unknown
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    date_created:              2025-09-05T08:03:27Z
    time_coverage_duration:    P8034D
    time_coverage_start:       1993-01-01T00:00:00Z
    time_coverage_end:         2014-12-31T00:00:00Z

In [19]:
edso1_dn23_sfcwind_c_cold

<xarray.Dataset>
Dimensions:                        (obs: 1102293, NbSample: 50)
Coordinates:
  * obs                            (obs) float64 8.0 9.0 ... 1.95e+06 1.95e+06
Dimensions without coordinates: NbSample
Data variables: (12/41)
    amplitude                      (obs) float64 -0.0442 -0.0326 ... -0.1393
    cost_association               (obs) float32 0.2921 0.2675 ... 9.969e+36
    effective_area                 (obs) float32 1.764e+09 ... 1.738e+10
    effective_contour_height       (obs) float32 0.056 0.044 ... -0.06 -0.06
    effective_contour_latitude     (obs, NbSample) float64 -69.76 ... -61.86
    effective_contour_longitude    (obs, NbSample) float64 355.2 355.2 ... 237.2
    ...                             ...
    dif_sfcwind                    (obs) float64 0.9812 0.8159 ... 0.09526
    sst_mean_ed                    (obs) float64 2.079 2.005 ... 5.442 5.523
    sst_npoints_ed                 (obs) float64 8.0 8.0 4.0 ... 51.0 48.0 48.0
    sst_mean_donut                 (obs) float64 2.201 2.053 ... 5.643 5.634
    sst_npoints_donut              (obs) float64 40.0 34.0 22.0 ... 238.0 254.0
    dif_sst                        (obs) float64 -0.1219 -0.04753 ... -0.1106
Attributes: (12/13)
    track_extra_variables:     lat_max,lon_max
    track_array_variables:     50
    array_variables:           contour_lat_e,contour_lon_e,contour_lat_s,cont...
    title:                     Cyclonic
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    ...                        ...
    framework_version:         0+unknown
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    date_created:              2025-09-05T08:03:27Z
    time_coverage_duration:    P8034D
    time_coverage_start:       1993-01-01T00:00:00Z
    time_coverage_end:         2014-12-31T00:00:00Z

#### select different radii

In [9]:
#>q75
ac1_q75 = edso1_dn23_sfcwind_ac["effective_radius"].quantile(0.75)
edso1_dn23_sfcwind_ac_q75 = edso1_dn23_sfcwind_ac.where(edso1_dn23_sfcwind_ac["effective_radius"] > ac1_q75, drop = True)
c1_q75 = edso1_dn23_sfcwind_c["effective_radius"].quantile(0.75)
edso1_dn23_sfcwind_c_q75 = edso1_dn23_sfcwind_c.where(edso1_dn23_sfcwind_c["effective_radius"] > c1_q75, drop = True)
print(ac1_q75, c1_q75)

<xarray.DataArray 'effective_radius' ()>
array(54150.)
Coordinates:
    quantile  float64 0.75 <xarray.DataArray 'effective_radius' ()>
array(52050.)
Coordinates:
    quantile  float64 0.75


In [10]:
#export data
edso1_dn23_sfcwind_ac_q75.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_ac_q75.nc")
edso1_dn23_sfcwind_c_q75.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_c_q75.nc")

In [22]:
#q50-q75
ac1_q50 = edso1_dn23_sfcwind_ac["effective_radius"].quantile(0.5)
edso1_dn23_sfcwind_ac_q50 = edso1_dn23_sfcwind_ac.where((edso1_dn23_sfcwind_ac["effective_radius"] < ac1_q75) & (edso1_dn23_sfcwind_ac["effective_radius"] > ac1_q50) , drop = True)
c1_q50 = edso1_dn23_sfcwind_c["effective_radius"].quantile(0.5)
edso1_dn23_sfcwind_c_q50 = edso1_dn23_sfcwind_c.where((edso1_dn23_sfcwind_c["effective_radius"] < c1_q75) & (edso1_dn23_sfcwind_c["effective_radius"] > c1_q50) , drop = True)
print(ac1_q50, c1_q50)

<xarray.DataArray 'effective_radius' ()>
array(39400.)
Coordinates:
    quantile  float64 0.5 <xarray.DataArray 'effective_radius' ()>
array(38350.)
Coordinates:
    quantile  float64 0.5


In [23]:
#q25-q50
ac1_q25 = edso1_dn23_sfcwind_ac["effective_radius"].quantile(0.25)
edso1_dn23_sfcwind_ac_q25 = edso1_dn23_sfcwind_ac.where((edso1_dn23_sfcwind_ac["effective_radius"] < ac1_q50) & (edso1_dn23_sfcwind_ac["effective_radius"] > ac1_q25) , drop = True)
c1_q25 = edso1_dn23_sfcwind_c["effective_radius"].quantile(0.25)
edso1_dn23_sfcwind_c_q25 = edso1_dn23_sfcwind_c.where((edso1_dn23_sfcwind_c["effective_radius"] < c1_q50) & (edso1_dn23_sfcwind_c["effective_radius"] > c1_q25) , drop = True)
print(ac1_q25, c1_q25)

<xarray.DataArray 'effective_radius' ()>
array(29150.)
Coordinates:
    quantile  float64 0.25 <xarray.DataArray 'effective_radius' ()>
array(28600.)
Coordinates:
    quantile  float64 0.25


In [24]:
#<q25
edso1_dn23_sfcwind_ac_q0 = edso1_dn23_sfcwind_ac.where(edso1_dn23_sfcwind_ac["effective_radius"] < ac1_q25, drop = True)
edso1_dn23_sfcwind_c_q0 = edso1_dn23_sfcwind_c.where(edso1_dn23_sfcwind_c["effective_radius"] < c1_q25, drop = True)

In [25]:
#export data
edso1_dn23_sfcwind_ac_q50.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_ac_q50.nc")
edso1_dn23_sfcwind_c_q50.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_c_q50.nc")

edso1_dn23_sfcwind_ac_q25.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_ac_q25.nc")
edso1_dn23_sfcwind_c_q25.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_c_q25.nc")

edso1_dn23_sfcwind_ac_q0.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_ac_q0.nc")
edso1_dn23_sfcwind_c_q0.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_c_q0.nc")

#### combine data

In [26]:
edso1_dn23_sfcwind_ac_q75["id"] = xr.DataArray(data=np.ones(edso1_dn23_sfcwind_ac_q75.dims["obs"]), dims=("obs",))
edso1_dn23_sfcwind_c_q75["id"] = xr.DataArray(data=np.zeros(edso1_dn23_sfcwind_c_q75.dims["obs"]), dims=("obs",))


In [27]:
edso1_dn23_sfcwind_q75 = xr.concat([edso1_dn23_sfcwind_ac_q75, edso1_dn23_sfcwind_c_q75], dim="obs")
edso1_dn23_sfcwind_q75 = edso1_dn23_sfcwind_q75.assign_coords(number=("obs", np.arange(edso1_dn23_sfcwind_q75.sizes["obs"])))


In [28]:
edso1_dn23_sfcwind_q75.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_q75.nc")

In [29]:
#add identification for ac = 1 and c = 0

edso1_dn23_sfcwind_ac_q50["id"] = xr.DataArray(data=np.ones(edso1_dn23_sfcwind_ac_q50.dims["obs"]), dims=("obs",))
edso1_dn23_sfcwind_c_q50["id"] = xr.DataArray(data=np.zeros(edso1_dn23_sfcwind_c_q50.dims["obs"]), dims=("obs",))

edso1_dn23_sfcwind_ac_q25["id"] = xr.DataArray(data=np.ones(edso1_dn23_sfcwind_ac_q25.dims["obs"]), dims=("obs",))
edso1_dn23_sfcwind_c_q25["id"] = xr.DataArray(data=np.zeros(edso1_dn23_sfcwind_c_q25.dims["obs"]), dims=("obs",))

edso1_dn23_sfcwind_ac_q0["id"] = xr.DataArray(data=np.ones(edso1_dn23_sfcwind_ac_q0.dims["obs"]), dims=("obs",))
edso1_dn23_sfcwind_c_q0["id"] = xr.DataArray(data=np.zeros(edso1_dn23_sfcwind_c_q0.dims["obs"]), dims=("obs",))

edso1_dn23_sfcwind_ac["id"] = xr.DataArray(data=np.ones(edso1_dn23_sfcwind_ac.dims["obs"]), dims=("obs",))
edso1_dn23_sfcwind_c["id"] = xr.DataArray(data=np.zeros(edso1_dn23_sfcwind_c.dims["obs"]), dims=("obs",))

In [30]:
#combine AC and C eddies
edso1_dn23_sfcwind_q50 = xr.concat([edso1_dn23_sfcwind_ac_q50, edso1_dn23_sfcwind_c_q50], dim="obs")
edso1_dn23_sfcwind_q50 = edso1_dn23_sfcwind_q50.assign_coords(number=("obs", np.arange(edso1_dn23_sfcwind_q50.sizes["obs"])))

edso1_dn23_sfcwind_q25 = xr.concat([edso1_dn23_sfcwind_ac_q25, edso1_dn23_sfcwind_c_q25], dim="obs")
edso1_dn23_sfcwind_q25 = edso1_dn23_sfcwind_q25.assign_coords(number=("obs", np.arange(edso1_dn23_sfcwind_q25.sizes["obs"])))

edso1_dn23_sfcwind_q0 = xr.concat([edso1_dn23_sfcwind_ac_q0, edso1_dn23_sfcwind_c_q0], dim="obs")
edso1_dn23_sfcwind_q0 = edso1_dn23_sfcwind_q0.assign_coords(number=("obs", np.arange(edso1_dn23_sfcwind_q0.sizes["obs"])))

edso1_dn23_sfcwind = xr.concat([edso1_dn23_sfcwind_ac, edso1_dn23_sfcwind_c], dim="obs")
edso1_dn23_sfcwind = edso1_dn23_sfcwind.assign_coords(number=("obs", np.arange(edso1_dn23_sfcwind.sizes["obs"])))

In [31]:
#export data
edso1_dn23_sfcwind_q50.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_q50.nc")

edso1_dn23_sfcwind_q25.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_q25.nc")

edso1_dn23_sfcwind_q0.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_q0.nc")

edso1_dn23_sfcwind.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind.nc")

#### shape error < e20

In [62]:
edso1_dn23_sfcwind_q75_e20 = edso1_dn23_sfcwind_q75.where(edso1_dn23_sfcwind_q75.effective_contour_shape_error < 20, drop = True)

In [63]:
#export data
edso1_dn23_sfcwind_q75_e20.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_q75_e20.nc")

In [12]:
edso1_sfcwind_ac_q75_e20 = edso1_sfcwind_ac_q75.where(edso1_sfcwind_ac_q75.effective_contour_shape_error < 20, drop = True)
edso1_sfcwind_c_q75_e20 = edso1_sfcwind_c_q75.where(edso1_sfcwind_c_q75.effective_contour_shape_error < 20, drop = True)

In [13]:
#export data
edso1_sfcwind_ac_q75_e20.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_sfcwind_ac_q75_e20.nc")
edso1_sfcwind_c_q75_e20.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_sfcwind_c_q75_e20.nc")

#### warm/cold

In [67]:
edso1_dn23_sfcwind_q75_e20_warm = edso1_dn23_sfcwind_q75_e20.where(edso1_dn23_sfcwind_q75_e20.dif_sst > 0, drop = True)
edso1_dn23_sfcwind_q75_e20_cold = edso1_dn23_sfcwind_q75_e20.where(edso1_dn23_sfcwind_q75_e20.dif_sst < 0, drop = True)

In [68]:
#export data
edso1_dn23_sfcwind_q75_e20_warm.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_q75_e20_warm.nc")
edso1_dn23_sfcwind_q75_e20_cold.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_q75_e20_cold.nc")

#### seasonality

In [69]:
edso1_dn23_sfcwind_q75_e20_jul = edso1_dn23_sfcwind_q75_e20.where(edso1_dn23_sfcwind_q75_e20["time"].dt.month == 7, drop = True)
edso1_dn23_sfcwind_q75_e20_jan = edso1_dn23_sfcwind_q75_e20.where(edso1_dn23_sfcwind_q75_e20["time"].dt.month == 1, drop = True)

In [70]:
#export data
edso1_dn23_sfcwind_q75_e20_jul.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_q75_e20_jul.nc")
edso1_dn23_sfcwind_q75_e20_jan.to_netcdf("/scratch/b/b383696/eddy_data/oceanS60/edso1_23/seaice3R/edso1_dn23_sfcwind_q75_e20_jan.nc")